# GPT-2 Native Perplexity Zero-Shot CEFR Classification

This notebook reproduces the GPT-2 native perplexity zero-shot experiment
from the `minimal-cefr` repository.

**Pipeline:**
1. Extract aggregate perplexity features from pre-trained GPT-2
2. Convert to numeric feature matrices
3. Train Logistic Regression + XGBoost classifiers on EFCAMDAT-train
4. Predict on 3 test sets (EFCAMDAT-test, CELVA-SP, KUPA-KEYS)
5. Generate ranked results summary

**Runtime:** ~20 min on T4 GPU with default limits, ~3h on CPU.

**Setup:** Go to `Runtime > Change runtime type > T4 GPU` before running.

---
## 0. Install Dependencies

In [ ]:
!pip install transformers torch xgboost scikit-learn pandas tqdm -q

---
## 1. Configuration

Set row limits and other experiment parameters.
Change any limit to `None` to use the full dataset (needed for paper reproduction).

In [ ]:
from pathlib import Path

# ── Row limits ────────────────────────────────────────────────────────
# Default: quick run (~20 min on GPU). Set to None for full reproduction.
LIMITS = {
    "norm-EFCAMDAT-train": 2000,   # Full: 80k (~40h CPU, ~2h GPU)
    "norm-EFCAMDAT-test":  500,    # Full: 20k (~10h CPU, ~30min GPU)
    "norm-CELVA-SP":       None,   # 1742 rows - always full
    "norm-KUPA-KEYS":      None,   # 1006 rows - always full
}

# ── Dataset roles ─────────────────────────────────────────────────────
DATASETS = {
    "norm-EFCAMDAT-train": "train",
    "norm-EFCAMDAT-test":  "test",
    "norm-CELVA-SP":       "test",
    "norm-KUPA-KEYS":      "test",
}

# ── Constants ─────────────────────────────────────────────────────────
CLASSIFIERS = ["logistic", "xgboost"]
CEFR_COL = "cefr_level"
FEATURE_DIR_NAME = "gpt2_native"

# ── Paths (Colab) ─────────────────────────────────────────────────────
REPO_DIR = Path("/content/minimal-cefr")
EXP_DIR = Path("/content/experiment")
PERPLEXITY_RAW = Path("/content/perplexity-raw")
TRIMMED_LABELS = Path("/content/trimmed-labels")

# DATA_PATH is set in the data acquisition step below
DATA_PATH = None  # will be set to a Path

print("Configuration loaded.")
print(f"  Limits: {LIMITS}")

---
## 2. Data Acquisition

Run **ONE** of the two options below (A or B), then continue.

### Option A: Google Drive

Use this if your data CSVs are in Google Drive.
Expected location: `MyDrive/phd-experimental-data/cefr-classification/data/splits/` containing:
- `norm-EFCAMDAT-train.csv`
- `norm-EFCAMDAT-test.csv`
- `norm-CELVA-SP.csv`
- `norm-KUPA-KEYS.csv`

In [ ]:
# ── OPTION A: Google Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Full paths for Colab (synced from i:/phd-experimental-data/cefr-classification/)
GDRIVE_DATA_ROOT = Path('/content/drive/MyDrive/phd-experimental-data/cefr-classification')
DATA_PATH = GDRIVE_DATA_ROOT / 'data' / 'splits'

print(f"DATA_PATH = {DATA_PATH}")

### Option B: Download via URL

Use this if you have a direct download link to a .zip file
containing the data CSVs.

In [ ]:
# ── OPTION B: wget + unzip ───────────────────────────────────────────
# Replace the URL below with your actual download link
DATA_URL = "YOUR_URL_HERE"  # <-- paste your .zip URL here

!wget -q --show-progress -O /content/cefr-data.zip "{DATA_URL}"
!unzip -q -o /content/cefr-data.zip -d /content/data

# Adjust this path to match the structure inside your zip
DATA_PATH = Path('/content/data/splits')

# If your zip extracts differently, list the contents to find the right path:
# !find /content/data -name '*.csv' | head -10

print(f"DATA_PATH = {DATA_PATH}")

### Verify Data

Check that all required files are present.

In [ ]:
assert DATA_PATH is not None, "Run Option A or Option B above first!"

missing = []
for name in DATASETS:
    f = DATA_PATH / f"{name}.csv"
    if f.exists():
        import pandas as pd
        n = len(pd.read_csv(f))
        print(f"  OK  {name}.csv ({n:,} rows)")
    else:
        print(f"  MISSING  {f}")
        missing.append(name)

if missing:
    raise FileNotFoundError(
        f"Missing files: {missing}. Check DATA_PATH or re-run data acquisition."
    )

---
## 3. Get Repository Code

Clone the `minimal-cefr` repository to get the `src/` modules.

If the repo is private, use **Alternative** below instead.

In [ ]:
# Clone the repository (change URL to your fork if needed)
!git clone https://github.com/YOUR_USERNAME/minimal-cefr.git /content/minimal-cefr 2>/dev/null || echo "Repository already cloned."

%cd /content/minimal-cefr

# Verify src/ exists
!ls src/*.py | head -5

**Alternative:** If you don't have a public repo, upload a zip of the `src/` directory:

```python
from google.colab import files
uploaded = files.upload()  # upload src.zip
!mkdir -p /content/minimal-cefr
!unzip -q -o src.zip -d /content/minimal-cefr
%cd /content/minimal-cefr
```

---
## 4. Experiment Setup

Create directory structure, copy data, detect GPU.

In [ ]:
import shutil
import torch

# ── Device detection ──────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("  WARNING: No GPU detected. Perplexity extraction will be slow.")
    print("  Go to Runtime > Change runtime type > T4 GPU")

# ── Create experiment directories ─────────────────────────────────────
for d in [
    EXP_DIR / "ml-training-data",
    EXP_DIR / "ml-test-data",
    EXP_DIR / "features",
    EXP_DIR / "feature-models",
    EXP_DIR / "results",
    PERPLEXITY_RAW,
    TRIMMED_LABELS,
]:
    d.mkdir(parents=True, exist_ok=True)

# ── Copy data into experiment structure ───────────────────────────────
for name, role in DATASETS.items():
    src = DATA_PATH / f"{name}.csv"
    if role == "train":
        dest = EXP_DIR / "ml-training-data" / f"{name}.csv"
    else:
        dest = EXP_DIR / "ml-test-data" / f"{name}.csv"

    if not dest.exists():
        shutil.copy2(src, dest)
        print(f"  Copied {name}.csv -> {dest.parent.name}/")
    else:
        print(f"  Already exists: {dest.name}")

print("\nSetup complete.")

---
## 5. Step 1+2: Extract GPT-2 Perplexity & Convert to Features

For each dataset:
1. Run GPT-2 perplexity extraction (GPU-accelerated)
2. Convert raw output to numeric feature matrix

**Runtime:** ~15 min on T4 GPU with default limits, ~2h on CPU.

In [ ]:
%%time
import subprocess
import sys
import pandas as pd

for name in DATASETS:
    limit = LIMITS.get(name)
    src_csv = DATA_PATH / f"{name}.csv"
    raw_out = PERPLEXITY_RAW / f"{name}.csv"

    # ── Step 1: Extract perplexity ────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  {name}  (limit={limit})")
    print(f"{'='*60}")

    if not raw_out.exists():
        cmd = [
            sys.executable, "-m", "src.extract_perplexity_features",
            "-i", str(src_csv),
            "--text-column", "text",
            "-m", "gpt2",
            "-d", DEVICE,
            "--aggregate-only",
            "-f", "csv",
            "-o", str(raw_out),
        ]
        if limit is not None:
            cmd.extend(["--limit", str(limit)])

        print(f"  Extracting perplexity...")
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=7200)
        if result.returncode != 0:
            print(f"  STDERR: {result.stderr[-2000:]}")
            raise RuntimeError(f"Perplexity extraction failed for {name}")
        # Show last few lines of output
        for line in result.stdout.strip().split('\n')[-5:]:
            print(f"  {line}")
    else:
        print(f"  Perplexity already extracted.")

    # ── Step 2: Convert to features_dense.csv ─────────────────────────
    feat_dir = EXP_DIR / "features" / FEATURE_DIR_NAME / name
    feat_dir.mkdir(parents=True, exist_ok=True)
    feat_out = feat_dir / "features_dense.csv"

    if not feat_out.exists():
        df = pd.read_csv(raw_out)
        drop_cols = [c for c in ["text", "model"] if c in df.columns]
        df_numeric = df.drop(columns=drop_cols)
        for col in df_numeric.columns:
            df_numeric[col] = pd.to_numeric(df_numeric[col], errors="coerce")
        df_numeric = df_numeric.fillna(0.0)
        df_numeric.to_csv(feat_out, index=False)

        fn_file = feat_dir / "feature_names.csv"
        pd.DataFrame({"feature_name": df_numeric.columns}).to_csv(fn_file, index=False)

        print(f"  Features: {len(df_numeric)} rows x {len(df_numeric.columns)} cols")
    else:
        print(f"  Features already converted.")

print("\nStep 1+2 complete.")

---
## 6. Step 3: Train Classifiers (LR + XGBoost)

Train on EFCAMDAT-train perplexity features.

**Runtime:** ~10 seconds.

In [ ]:
%%time
import subprocess
import sys
import pandas as pd

model_names = {}

features_file = (
    EXP_DIR / "features" / FEATURE_DIR_NAME / "norm-EFCAMDAT-train" / "features_dense.csv"
)
labels_csv = EXP_DIR / "ml-training-data" / "norm-EFCAMDAT-train.csv"

# Check if labels need trimming (due to --limit)
n_features = len(pd.read_csv(features_file))
df_labels = pd.read_csv(labels_csv)
if len(df_labels) != n_features:
    print(f"  Trimming labels from {len(df_labels)} to {n_features} rows")
    trimmed = TRIMMED_LABELS / "norm-EFCAMDAT-train.csv"
    df_labels.head(n_features).to_csv(trimmed, index=False)
    labels_csv = trimmed

for clf_type in CLASSIFIERS:
    model_name = f"norm-EFCAMDAT-train_{clf_type}_gpt2native"
    model_names[clf_type] = model_name
    model_dir = EXP_DIR / "feature-models" / "classifiers" / model_name

    if (model_dir / "classifier.pkl").exists():
        print(f"  Already trained: {model_name}")
        continue

    print(f"\n  Training {clf_type}...")
    cmd = [
        sys.executable, "-m", "src.train_classifiers",
        "-e", str(EXP_DIR),
        "--features-file", str(features_file),
        "--labels-csv", str(labels_csv),
        "--cefr-column", CEFR_COL,
        "--classifier", clf_type,
        "--model-name", model_name,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  STDERR: {result.stderr[-2000:]}")
        raise RuntimeError(f"Training failed for {clf_type}")
    for line in result.stdout.strip().split('\n')[-5:]:
        print(f"  {line}")

print(f"\nTrained models: {list(model_names.values())}")

---
## 7. Step 4: Predict on Test Sets

Run each classifier on all 3 test sets (6 predictions total).

**Runtime:** ~10 seconds.

In [ ]:
%%time
import subprocess
import sys
import pandas as pd

test_sets = [n for n, role in DATASETS.items() if role == "test"]

for clf_type in CLASSIFIERS:
    model_name = model_names[clf_type]
    for test_name in test_sets:
        results_dir = EXP_DIR / "results" / model_name / test_name

        if (results_dir / "evaluation_report.md").exists():
            print(f"  Already done: {clf_type} -> {test_name}")
            continue

        features_file = (
            EXP_DIR / "features" / FEATURE_DIR_NAME / test_name / "features_dense.csv"
        )
        labels_csv = EXP_DIR / "ml-test-data" / f"{test_name}.csv"

        # Check if labels need trimming
        n_features = len(pd.read_csv(features_file))
        df_labels = pd.read_csv(labels_csv)
        if len(df_labels) != n_features:
            print(f"  Trimming {test_name} labels: {len(df_labels)} -> {n_features}")
            trimmed = TRIMMED_LABELS / f"{test_name}.csv"
            df_labels.head(n_features).to_csv(trimmed, index=False)
            labels_csv = trimmed

        print(f"  Predicting: {clf_type} -> {test_name}")
        cmd = [
            sys.executable, "-m", "src.predict",
            "-e", str(EXP_DIR),
            "-m", model_name,
            "--features-file", str(features_file),
            "--labels-csv", str(labels_csv),
            "--cefr-column", CEFR_COL,
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            print(f"  STDERR: {result.stderr[-2000:]}")
            raise RuntimeError(f"Prediction failed: {model_name} -> {test_name}")
        for line in result.stdout.strip().split('\n')[-3:]:
            print(f"    {line}")

print("\nAll predictions complete.")

---
## 8. Step 5: Generate Report

Aggregate all evaluation results into a ranked summary table.

In [ ]:
import subprocess
import sys

summary_path = EXP_DIR / "results_summary.md"

cmd = [
    sys.executable, "-m", "src.report",
    "-e", str(EXP_DIR),
    "--rank", "accuracy",
    "--summary-report", str(summary_path),
    "--include-all-datasets",
    "-v",
]
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode != 0:
    print(f"STDERR: {result.stderr[-2000:]}")
    raise RuntimeError("Report generation failed")
print(result.stdout[-1000:])

---
## 9. Results

Display the results summary.

In [ ]:
from IPython.display import Markdown, display

summary_path = EXP_DIR / "results_summary.md"

if summary_path.exists():
    display(Markdown(summary_path.read_text()))
else:
    print("No results summary found. Run Step 5 above.")

### Individual Evaluation Reports

View detailed per-model, per-dataset evaluation reports.

In [ ]:
from IPython.display import Markdown, display

for clf_type in CLASSIFIERS:
    model_name = model_names[clf_type]
    test_sets = [n for n, role in DATASETS.items() if role == "test"]
    for test_name in test_sets:
        report_path = EXP_DIR / "results" / model_name / test_name / "evaluation_report.md"
        if report_path.exists():
            display(Markdown(f"---\n### {clf_type} on {test_name}"))
            display(Markdown(report_path.read_text()))
        else:
            print(f"  Missing: {report_path}")

---
## 10. Download Results (Optional)

Download the experiment outputs to your local machine.

In [ ]:
# Zip the experiment directory for download
!cd /content && zip -r -q experiment-results.zip experiment/results/ experiment/results_summary.md

from google.colab import files
files.download('/content/experiment-results.zip')